In [3]:
import torch
import torch.nn as nn
from model import UNet

# Load the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet(1,1).to(device)

model_path = "models/model_10.pth"  # Update with correct path
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()


UNet(
  (down): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (up): Up(
    (up): Upsample(scale_factor=2.0, mode='bilinear')
  )
  (out): Conv2d(64, 1, kernel_size=(1, 1), stride=(1, 1))
  (right1): DoubleConv(
    (conv1): Sequential(
      (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
    (conv2): Sequential(
      (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
  )
  (right2): DoubleConv(
    (conv1): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
    (

In [2]:
# # Load the dataset
# from dataset import load_datasets
# train_d, test_d = load_datasets('CBSD68-dataset/CBSD68/noisy35', 'CBSD68-dataset/CBSD68/original_png')

# import torch
# import numpy as np
# from skimage.metrics import peak_signal_noise_ratio, structural_similarity

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# from torchmetrics.image.ssim import StructuralSimilarityIndexMeasure
# from skimage.metrics import peak_signal_noise_ratio

# # Initialize SSIM metric (Move to GPU if available)
# ssim_metric = StructuralSimilarityIndexMeasure().to("cuda" if torch.cuda.is_available() else "cpu")

# def calculate_metrics(img1, img2):
#     """Calculates PSNR and SSIM between two images.

#     Args:
#         img1 (torch.Tensor or np.ndarray): Noisy/Denoised image.
#         img2 (torch.Tensor or np.ndarray): Clean image.

#     Returns:
#         dict: Dictionary with PSNR and SSIM values.
#     """
#     # Convert to torch tensor if needed
#     if isinstance(img1, np.ndarray):
#         img1 = torch.tensor(img1).unsqueeze(0)  # Add batch dimension
#     if isinstance(img2, np.ndarray):
#         img2 = torch.tensor(img2).unsqueeze(0)

#     # Ensure both are on the same device
#     device = "cuda" if torch.cuda.is_available() else "cpu"
#     img1, img2 = img1.to(device), img2.to(device)

#     # Compute PSNR
#     psnr = peak_signal_noise_ratio(img2.cpu().numpy(), img1.cpu().numpy(), data_range=1.0)

#     # Compute SSIM using torchmetrics
#     ssim = ssim_metric(img1, img2).item()

#     return {"PSNR": psnr, "SSIM": ssim}


# Load a single noisy-clean image pair
noisy_img, clean_img = train_d[0]
noisy_img, clean_img = noisy_img.to(device), clean_img.to(device)

# Add batch dimension
noisy_img = noisy_img.unsqueeze(0)
clean_img = clean_img.unsqueeze(0)

# Denoise the image
with torch.no_grad():
    denoised_img = model(noisy_img)

# Remove batch dimension
denoised_img = denoised_img.squeeze(0)
noisy_img = noisy_img.squeeze(0)
clean_img = clean_img.squeeze(0)

# Convert to NumPy (Ensure correct format)
noisy_np = noisy_img.cpu().numpy()
clean_np = clean_img.cpu().numpy()
denoised_np = denoised_img.cpu().numpy()

# Calculate PSNR & SSIM
metrics_noisy = calculate_metrics(noisy_np, clean_np)
metrics_denoised = calculate_metrics(denoised_np, clean_np)

print(f"PSNR (Noisy vs Clean): {metrics_noisy['PSNR']:.2f}, SSIM: {metrics_noisy['SSIM']:.4f}")
print(f"PSNR (Denoised vs Clean): {metrics_denoised['PSNR']:.2f}, SSIM: {metrics_denoised['SSIM']:.4f}")

PSNR (Noisy vs Clean): 20.93, SSIM: 0.2544
PSNR (Denoised vs Clean): 22.36, SSIM: 0.5704


NameError: name 'show' is not defined